# Fake News Detection - Training Demo

This notebook demonstrates how to train the fake news detection models using the provided framework.

In [ ]:
# Import necessary modules
import sys
import os
sys.path.append('../src')

from preprocessing import prepare_sample_data, preprocess_dataset, split_data
from training import FakeNewsTrainer, get_default_param_grids
import pandas as pd
import numpy as np

## 1. Data Preparation

In [ ]:
# Load sample data
print("Loading sample data...")
df = prepare_sample_data()
print(f"Dataset shape: {df.shape}")
print("\nSample data:")
df.head()

In [ ]:
# Preprocess the data
print("Preprocessing data...")
X, y = preprocess_dataset(df)
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 2. Model Training

In [ ]:
# Initialize trainer
trainer = FakeNewsTrainer()
param_grids = get_default_param_grids()

print("Available models:", list(trainer.models.keys()))

In [ ]:
# Train all models
print("Training models...")
trainer.train_all_models(X_train, y_train, param_grids, cv_folds=3)

## 3. Model Evaluation

In [ ]:
# Evaluate models
print("Evaluating models...")
results_df = trainer.evaluate_all_models(X_test, y_test)
print("\nEvaluation Results:")
results_df

In [ ]:
# Plot confusion matrices
for model_name in trainer.trained_models.keys():
    trainer.plot_confusion_matrix(model_name, y_test)

## 4. Feature Analysis

In [ ]:
# Get feature importance for Random Forest
try:
    feature_importance = trainer.get_feature_importance('random_forest')
    print("Top 10 most important features:")
    feature_importance.head(10)
except Exception as e:
    print(f"Feature importance analysis failed: {e}")

## 5. Cross-Validation

In [ ]:
# Perform cross-validation
for model_name in trainer.trained_models.keys():
    cv_results = trainer.cross_validate_model(model_name, X, y, cv_folds=5)
    print(f"\n{model_name} Cross-Validation:")
    print(f"Mean Accuracy: {cv_results['mean_accuracy']:.4f} (+/- {cv_results['std_accuracy']*2:.4f})")

## 6. Save Models

In [ ]:
# Save trained models
trainer.save_models()
print("Models saved successfully!")

## 7. Test Predictions

In [ ]:
# Test with new examples
from prediction import FakeNewsPredictor

predictor = FakeNewsPredictor()

test_texts = [
    "Scientists at MIT have developed a new renewable energy technology.",
    "SHOCKING: This miracle cure will heal any disease instantly!",
    "The stock market closed higher today following positive economic data."
]

for i, text in enumerate(test_texts, 1):
    result = predictor.predict_single(text)
    print(f"\nText {i}: {text}")
    print(f"Prediction: {result['label']} (Confidence: {result['confidence']:.2f})")